[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Column Types &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and makes what its worked
examples made: `Method`, `Student` and `Payment`, the `payments` table, `record_payment` and
`statement`, and `Fee` on its `TypedBase`. Run it first. Task 3 refunds the payment that task 2
records, and the last cell removes the scratch folder.


In [1]:
import enum
import shutil
import uuid
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, Column, Date, DateTime, Enum, ForeignKey, Integer, MetaData, Numeric, String,
                        Table, UniqueConstraint, create_engine, event, func, insert, select, text)
from sqlalchemy.dialects import mysql, postgresql
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column
from sqlalchemy.pool import StaticPool
from sqlalchemy.schema import CreateTable
from sqlalchemy import Float

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}
college = MetaData(naming_convention=NAMING)

students = Table(
    "students", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(100), nullable=False),
    Column("email", String(200), nullable=False, unique=True),
    Column("program", String(50), nullable=False),
    Column("started_on", Date, nullable=False),
)
courses = Table(
    "courses", college,
    Column("id", Integer, primary_key=True),
    Column("code", String(10), nullable=False, unique=True),
    Column("title", String(100), nullable=False),
    Column("department", String(50), nullable=False),
    Column("credits", Integer, nullable=False),
    CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),
)
terms = Table(
    "terms", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(20), nullable=False, unique=True),
    Column("starts_on", Date, nullable=False),
)
sections = Table(
    "sections", college,
    Column("id", Integer, primary_key=True),
    Column("course_id", ForeignKey("courses.id"), nullable=False),
    Column("term_id", ForeignKey("terms.id"), nullable=False),
    Column("capacity", Integer, nullable=False),
    UniqueConstraint("course_id", "term_id"),
    CheckConstraint("capacity > 0", name="capacity_positive"),
)
enrollments = Table(
    "enrollments", college,
    Column("student_id", ForeignKey("students.id"), primary_key=True),
    Column("section_id", ForeignKey("sections.id"), primary_key=True),
    Column("status", String(20), nullable=False, server_default="enrolled"),
    Column("grade", String(2)),
    CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),
)

def build_college(engine):
    """Create the college's tables from `college`, load the lists from Setup into them, and count their rows."""
    college.create_all(engine)
    rows = {
        students: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                   for name, email, program, started in STUDENTS],
        courses: [{"code": code, "title": title, "department": department, "credits": credits}
                  for code, title, department, credits in COURSES],
        terms: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        sections: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        enrollments: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                      for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for table in college.sorted_tables:
            conn.execute(insert(table), rows[table])
        return {table.name: conn.execute(select(func.count()).select_from(table)).scalar_one()
                for table in college.sorted_tables}

def show_ddl(table, dialect):
    """Print the CREATE TABLE statement a Table becomes in one database's dialect."""
    for line in str(CreateTable(table).compile(dialect=dialect)).strip().splitlines():
        print("   ", line.rstrip().replace("\t", "    "))

engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))


class Method(enum.Enum):
    CARD = "card"
    TRANSFER = "transfer"
    CASH = "cash"


class Base(DeclarativeBase):
    pass


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))


class Payment(Base):
    __tablename__ = "payments"

    id: Mapped[int] = mapped_column(primary_key=True)
    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"))
    amount: Mapped[Decimal] = mapped_column(Numeric(10, 2))
    received_at: Mapped[datetime]
    method: Mapped[Method]
    receipt: Mapped[uuid.UUID]
    refunded: Mapped[bool] = mapped_column(default=False)
    note: Mapped[str | None]

    def __repr__(self):
        return f"Payment({self.amount}, {self.method.name}, {self.received_at:%Y-%m-%d %H:%M})"


Base.metadata.create_all(engine)

def record_payment(session, student_id, amount, paid_at, method, number):
    """Record a payment from a form's text: an amount, a time with its offset, a method, and a receipt number."""
    payment = Payment(
        student_id=student_id,
        amount=Decimal(amount).quantize(Decimal("0.01")),
        received_at=datetime.fromisoformat(paid_at).astimezone(timezone.utc).replace(tzinfo=None),
        method=Method(method),
        receipt=uuid.uuid5(uuid.NAMESPACE_URL, f"https://college.edu/receipts/{number}"),
    )
    session.add(payment)
    return payment


def statement(session, student_id, tuition):
    """A student's payments, oldest first, with their times in UTC, and the tuition still owed, in Decimal."""
    payments = session.scalars(
        select(Payment).where(Payment.student_id == student_id, Payment.refunded.is_(False)).order_by(Payment.received_at)
    ).all()
    lines = [(payment.received_at.replace(tzinfo=timezone.utc).isoformat(), payment.method.value, payment.amount)
             for payment in payments]
    return lines, tuition - sum((payment.amount for payment in payments), Decimal("0.00"))


class TypedBase(DeclarativeBase):
    type_annotation_map = {str: String(100), Decimal: Numeric(10, 2), datetime: DateTime(timezone=True)}


class Fee(TypedBase):
    __tablename__ = "fees"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    amount: Mapped[Decimal]
    due: Mapped[datetime]


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


**1.** The Python type behind every column.


In [2]:
for column in Payment.__table__.columns:
    print(f"{column.name:<12} {column.type.python_type}")


id           <class 'int'>
student_id   <class 'int'>
amount       <class 'decimal.Decimal'>
received_at  <class 'datetime.datetime'>
method       <enum 'Method'>
receipt      <class 'uuid.UUID'>
refunded     <class 'bool'>
note         <class 'str'>


`python_type` is the type a column returns, and for `method` it is the enum class itself, `Method`.


**2.** A payment from a form, one hour ahead of UTC.


In [3]:
with Session(engine) as session:
    transfer = record_payment(session, 2, "500", "2026-01-20T15:45+01:00", "transfer", 6)
    session.commit()
    print(repr(transfer.amount), "|", repr(transfer.received_at), "|", transfer.method)


Decimal('500.00') | datetime.datetime(2026, 1, 20, 14, 45) | Method.TRANSFER


15:45 at one hour ahead of UTC is 14:45 in UTC, which is what was stored, with no zone, and `"500"`
became `Decimal('500.00')`.


**3.** A refund, and what SQLite stores for it.


In [4]:
with Session(engine) as session:
    payment = session.scalars(select(Payment).where(Payment.student_id == 2)).one()
    payment.refunded = True
    session.commit()

with engine.connect() as conn:
    print(conn.execute(text("SELECT refunded, typeof(refunded) FROM payments WHERE student_id = 2")).one())


(1, 'integer')


`True` went into SQLite as the integer 1. Changing an attribute and committing is how the ORM updates
a row, which **The Session** notebook explains.


**4.** `Float` against `Numeric`.


In [5]:
class MeasureBase(DeclarativeBase):
    pass


class Measure(MeasureBase):
    __tablename__ = "measures"

    id: Mapped[int] = mapped_column(primary_key=True)
    approximate: Mapped[float] = mapped_column(Float)
    exact: Mapped[Decimal] = mapped_column(Numeric(10, 2))


measures = college_engine()
MeasureBase.metadata.create_all(measures)
with Session(measures) as session:
    session.add(Measure(approximate=0.1 + 0.2, exact=Decimal("0.1") + Decimal("0.2")))
    session.commit()
    stored = session.scalars(select(Measure)).one()
    print(repr(stored.approximate), "|", repr(stored.exact))
measures.dispose()


0.30000000000000004 | Decimal('0.30')


`0.1 + 0.2` in floating point is `0.30000000000000004`, and the `Float` column kept exactly that. The
`Decimal` sum is exactly `0.3`, and the `Numeric` column returned it with its two places.


**5.** An enum that stores values.


In [6]:
class Status(enum.Enum):
    ENROLLED = "enrolled"
    COMPLETED = "completed"
    WITHDRAWN = "withdrawn"


class EnrollmentBase(DeclarativeBase):
    pass


class Enrollment(EnrollmentBase):
    __tablename__ = "enrollments"

    student_id: Mapped[int] = mapped_column(primary_key=True)
    section_id: Mapped[int] = mapped_column(primary_key=True)
    status: Mapped[Status] = mapped_column(Enum(Status, values_callable=lambda members: [m.value for m in members],
                                                length=20))


with Session(engine) as session:
    print(session.scalar(select(func.count()).select_from(Enrollment).where(Enrollment.status == Status.ENROLLED)))


75


Seventy-five, the Spring 2026 enrollments, all still under way. The condition compared with
`'enrolled'`, the value, because `values_callable` made the values the text that stands for each
member.


**6.** One class, two databases.


In [7]:
print("PostgreSQL:")
show_ddl(Fee.__table__, postgresql.dialect())
print("MySQL:")
show_ddl(Fee.__table__, mysql.dialect())


PostgreSQL:
    CREATE TABLE fees (
        id SERIAL NOT NULL,
        name VARCHAR(100) NOT NULL,
        amount NUMERIC(10, 2) NOT NULL,
        due TIMESTAMP WITH TIME ZONE NOT NULL,
        PRIMARY KEY (id)
    )
MySQL:
    CREATE TABLE fees (
        id INTEGER NOT NULL AUTO_INCREMENT,
        name VARCHAR(100) NOT NULL,
        amount NUMERIC(10, 2) NOT NULL,
        due DATETIME NOT NULL,
        PRIMARY KEY (id)
    )


PostgreSQL keeps the zone, in `TIMESTAMP WITH TIME ZONE`. MySQL's `DATETIME` has none, the same as
SQLite, so a program on MySQL needs the same habit of storing UTC. Both accepted the strings, since
`type_annotation_map` gave every one of them a length.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Column Types](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/08-column-types.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
